In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.chains.retrieval_qa.base import RetrievalQA
from langchain.document_loaders import CSVLoader
from langchain.vectorstores import DocArrayInMemorySearch, DocArrayHnswSearch
from langchain.indexes.vectorstore import VectorstoreIndexCreator
from IPython.display import display, Markdown

In [2]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    # model="qwen3:1.7B",
    temperature=0.9,
    verbose=True,
    extract_reasoning=True,
)
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

In [3]:
loader = CSVLoader(file_path="../data/04-OutdoorClothingCatalog_1000.csv")

In [4]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

/home/eugene/projects/deeplearning.ai/.venv/lib/python3.13/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


In [5]:
query = "Please list all your shirts with sun protection in a table in markdown and summarize each one"

In [6]:
response = index.query(question=query, llm=llm)
display(Markdown(response))



| **Shirt Name**                              | **UPF Rating** | **Fabric Composition**                          | **Care Instructions**         | **Key Features**                                                                                                                                 |
|---------------------------------------------|----------------|------------------------------------------------|-------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------|
| **Sun Shield Shirt**                        | UPF 50+        | 78% nylon, 22% Lycra Xtra Life               | Handwash, line dry           | High-performance, blocks 98% UV rays; abrasion-resistant; fits over swimsuit; moisture-wicking.                                                  |
| **Men's Plaid Tropic Shirt, Short-Sleeve**  | UPF 50+        | 52% polyester, 48% nylon                    | Machine washable, dryable    | SunSmart technology blocks 98% UV; front/back cape vents; two front bellows pockets; wrinkle-free.                                               |
| **Men's Tropical Plaid Short-Sleeve Shirt** | UPF 50+        | 100% polyester                              | Machine wash and dry         | Traditional relaxed fit; front/back cape vents; two front bellows pockets; wrinkle-resistant; imported.                                           |
| **Women's Tropical Tee, Sleeveless**        | UPF 50+        | Shell: 71% nylon, 29% polyester; Cape: 100% polyester | Machine wash and dry         | Five-star sleeveless button-up; built-in SunSmart UPF 50+; updated design; low-profile pockets; eyewear loop; side shaping.                    |

**Summary:**  
All four shirts feature UPF 50+ sun protection, blocking 98% of UV rays. The Sun Shield Shirt uses nylon/Lycra for durability, while the Tropical Plaid Shirts (men’s and women’s) rely on 100% polyester for wrinkle resistance. The Plaid Tropic Shirt includes practical features like cape vents and pockets, while the Women’s Tropical Tee emphasizes a flattering fit with built-in sun protection. Care instructions vary, with some requiring handwashing and others being machine-friendly.

In [13]:
documents = loader.load()
db = DocArrayHnswSearch.from_documents(
    documents=documents,
    embedding=embeddings,
    work_dir="../data/temp/",
    n_dim=768,
)

In [14]:
retriever = db.as_retriever()
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    verbose=True,
)
response = qa.invoke(query)
display(Markdown(response["result"]))



> Entering new RetrievalQA chain...

> Finished chain.




| Name | Description | UPF Rating | Key Features |
|------|-------------|------------|--------------|
| **Sun Shield Shirt** | High-performance sun shirt with UPF 50+ protection, blocking 98% of UV rays. Slightly fitted, made of 78% nylon and 22% Lycra. Handwash, line dry. | UPF 50+ | UV protection, abrasion-resistant, fits over swimsuit. |
| **Men's Plaid Tropic Shirt** | UPF 50+ rated shirt with SunSmart technology. Lightest hot-weather shirt, 52% polyester, 48% nylon. Front/back cape venting, two bellows pockets. | UPF 50+ | Wrinkle-free, quick-dry, designed for fishing/travel. |
| **Men's Tropical Plaid Short-Sleeve Shirt** | UPF 50+ rated, 100% polyester. Traditional fit with relaxed chest, sleeves, and waist. Front/back cape venting, two bellows pockets. | UPF 50+ | Lightweight, wrinkle-resistant, imported. |
| **Sunrise Tee** | Women's UV-protective shirt with UPF 50+ (SunSmart™). Lightweight, moisture-wicking fabric. Includes pockets, eyewear loop, and front/back cape venting. | UPF 50+ | Built-in sun protection, wrinkle-free, machine washable. |

**Summary**: All four shirts offer UPF 50+ sun protection, blocking 98% of UV rays. The Sun Shield Shirt and Sunrise Tee emphasize UV protection that doesn’t wear off, while the Tropical Plaid and Tropic Shirt focus on lightweight, breathable designs. The Tropic Shirt is ideal for active use, and the Sunrise Tee includes practical features like pockets and a eyewear loop.